# Figure 7 | Llano data diagnosis and reinversion

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig7/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

The TEM processing panel documents normalization, but the selected TEM inversion was vetoed retrospectively. No model approval is implied by this panel.

## Setup and saved ERT/SRT runs

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap, BoundaryNorm
from matplotlib.collections import PolyCollection
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from pyproj import Transformer
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
CASE = ROOT / 'Result' / 'Llano-Uplift'
OUT = ROOT / 'examples' / 'Figure' / 'fig7'
OUT.mkdir(parents=True, exist_ok=True)
STATE = json.loads((CASE / 'case_state.json').read_text(encoding='utf-8'))

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'axes.linewidth': 1.1,
    'axes.grid': False, 'legend.frameon': False,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
})

RHO_CMAP = mpl.colormaps['turbo'].copy()
RHO_CMAP.set_bad('white')
RHO_NORM = LogNorm(30, 2000)
RHO_TICKS = [30, 100, 1000]
VEL_CMAP = mpl.colormaps['viridis'].copy()
VEL_NORM = mpl.colors.Normalize(500, 2200)
UNIT_COLORS = ['#FFFFFF', '#CF604A', '#4C78A8', '#73A5A1', '#D8DCE1']
UNIT_CMAP = ListedColormap(UNIT_COLORS)
UNIT_NORM = BoundaryNorm(np.arange(-0.5, 5.5), UNIT_CMAP.N)

def bold_axes(ax):
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def save_png(fig, name):
    target = OUT / name
    fig.savefig(target, dpi=400, bbox_inches='tight', pad_inches=0.12,
                facecolor='white')
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = __import__('os').environ.get('FIG7_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(name).with_suffix('.pdf'), bbox_inches='tight', pad_inches=0.12,
                    facecolor='white')
    plt.close(fig)
    return target

def vertical_colorbar(fig, cax, mappable, label, ticks):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks(ticks)
    cb.set_ticklabels([str(int(tick)) for tick in ticks])
    cb.set_label(label, fontsize=13, fontweight='bold', labelpad=9)
    cb.ax.tick_params(width=1.2, labelsize=11)
    for tick in cb.ax.get_yticklabels():
        tick.set_fontweight('bold')
    return cb


In [ ]:
def load_mesh_run(run_id):
    folder = CASE / 'runs' / run_id
    mesh = np.load(folder / 'mesh_geometry.npz')
    result = json.loads((folder / 'result.json').read_text(encoding='utf-8'))
    return folder, mesh, result

ERT = {}
for item in STATE['ert_imports']:
    name = f"ERT{item['profile']}"
    folder, mesh, result = load_mesh_run(STATE['ert_runs'][name])
    ERT[name] = {
        'mesh': mesh, 'result': result,
        'electrodes': np.load(ROOT / item['electrodes_path']),
        'values': np.load(folder / 'recovered_model.npy'),
        'coverage': np.load(folder / 'model_coverage.npy'),
    }

srt_item = STATE['srt_imports'][0]
assert srt_item['profile'] == 2 and list(STATE['srt_runs']) == ['SRT2']
srt_folder, srt_mesh, srt_result = load_mesh_run(STATE['srt_runs']['SRT2'])
SRT2 = {
    'mesh': srt_mesh, 'result': srt_result,
    'sensors': np.load(ROOT / srt_item['sensors_path']),
    'values': np.load(srt_folder / 'recovered_model.npy'),
    'touched': np.load(srt_folder / 'cells_touched_by_a_ray.npy').astype(bool),
}
tdem_xy = np.asarray(STATE['tdem_import']['location_m'][:2], float)



## Geometry and section drawing

In [ ]:
from scipy.interpolate import griddata
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle
import matplotlib.patheffects as pe

utm = Transformer.from_crs('EPSG:4269', 'EPSG:26914', always_xy=True)

def survey_lines(path, id_column):
    table = np.genfromtxt(path, delimiter='\t', names=True, encoding='utf-8')
    lines = {}
    for profile in np.unique(table[id_column]):
        rows = table[table[id_column] == profile]
        distance, first = np.unique(rows['Profile_x_Coordinate_meters'], return_index=True)
        x, y = utm.transform(rows['Longitude'][first], rows['Latitude'][first])
        lines[int(profile)] = {'distance': distance, 'xy': np.c_[x, y]}
    return lines

ERT_LINES = survey_lines(ROOT / 'data' / 'case3_Llano' / 'ert' / 'electrodes_xyz.txt', 'ERT_Profile_ID')
SRT_LINES = survey_lines(ROOT / 'data' / 'case3_Llano' / 'srt' / 'geometry.txt', 'SRT_Profile_ID')

def along(line, point):
    # Distance along a surveyed line of its nearest point to `point`, and how far off it.
    s = np.linspace(line['distance'][0], line['distance'][-1], 2001)
    xy = np.c_[np.interp(s, line['distance'], line['xy'][:, 0]), np.interp(s, line['distance'], line['xy'][:, 1])]
    k = np.argmin(np.linalg.norm(xy - point, axis=1))
    return float(s[k]), float(np.linalg.norm(xy[k] - point))

def crossing(first, second):
    s1 = np.linspace(first['distance'][0], first['distance'][-1], 1501)
    s2 = np.linspace(second['distance'][0], second['distance'][-1], 1501)
    p1 = np.c_[np.interp(s1, first['distance'], first['xy'][:, 0]), np.interp(s1, first['distance'], first['xy'][:, 1])]
    p2 = np.c_[np.interp(s2, second['distance'], second['xy'][:, 0]), np.interp(s2, second['distance'], second['xy'][:, 1])]
    d = np.linalg.norm(p1[:, None] - p2[None], axis=2)
    i, j = np.unravel_index(np.argmin(d), d.shape)
    return float(s1[i]), float(s2[j])

COLUMN_ON_ERT2, _ = along(ERT_LINES[2], tdem_xy)
COLUMN_ON_SRT2, _ = along(SRT_LINES[2], tdem_xy)
ERT2_MEETS_SRT2, SRT2_MEETS_ERT2 = crossing(ERT_LINES[2], SRT_LINES[2])
ERT2_MEETS_ERT1, _ = crossing(ERT_LINES[2], ERT_LINES[1])
_, SRT2_MEETS_ERT1 = crossing(ERT_LINES[1], SRT_LINES[2])
print(f'column: ERT2 x = {COLUMN_ON_ERT2:.1f} m, SRT2 x = {COLUMN_ON_SRT2:.1f} m; '
      f'SRT2 crosses ERT2 at {SRT2_MEETS_ERT2:.1f} m and ERT1 at {SRT2_MEETS_ERT1:.1f} m')

COVERAGE_BAR_DECADES = 2.0
ERT_COLOR, EM_COLOR, SRT_COLOR = '#C8553D', '#2A5C8A', '#6F4E7C'

def reliable_ert(name):
    item = ERT[name]
    return (item['coverage'] > np.nanmax(item['coverage']) - COVERAGE_BAR_DECADES) & (item['values'] > 0)

def section_image(centres, values, keep, surface_x, surface_z, xlim, zlim, log=True, step=0.25):
    # Display interpolation of a mesh model, blanked where the model is not kept and above ground.
    xg = np.arange(xlim[0], xlim[1] + step / 2, step)
    zg = np.arange(zlim[0], zlim[1] + step / 2, step)
    X, Z = np.meshgrid(xg, zg)
    v = np.log10(values) if log else np.asarray(values, float)
    linear = griddata(centres, v, (X, Z), method='linear')
    kept = griddata(centres, keep.astype(float), (X, Z), method='nearest') > 0.5
    ground = np.interp(xg, surface_x, surface_z)
    image = np.where(np.isfinite(linear) & kept & (Z <= ground[None, :]), linear, np.nan)
    return xg, zg, (10.0 ** image) if log else image

def column_ring(ax, x, z):
    ax.scatter([x], [z], s=58, facecolor='none', edgecolor='white', linewidth=0.8, zorder=11, clip_on=False)
    ax.scatter([x], [z], s=34, facecolor='none', edgecolor='#111111', linewidth=1.1, zorder=12, clip_on=False)

def crossing_tick(ax, x, z, label, ha):
    ax.plot([x, x], [z, z + 1.6], color='#111111', lw=1.2, zorder=10, clip_on=False)
    ax.text(x + (0.8 if ha == 'left' else -0.8), z + 1.9, label, ha=ha, va='bottom',
            fontsize=10, fontweight='bold', zorder=10, clip_on=False)

def draw_ert_section(name, filename, marks, brackets=()):
    item = ERT[name]
    centres = np.asarray(item['mesh']['cell_centers_m'], float)[:, :2]
    electrodes = item['electrodes']
    xg, zg, image = section_image(centres, item['values'], reliable_ert(name),
                                  electrodes[:, 0], electrodes[:, 2], (0, 188), (267, 300))
    fig = plt.figure(figsize=(9.2, 2.7))
    ax = fig.add_axes([0.10, 0.22, 0.70, 0.56])
    cax = fig.add_axes([0.83, 0.22, 0.028, 0.56])
    mesh = ax.pcolormesh(xg, zg, image, cmap=RHO_CMAP, norm=RHO_NORM, shading='nearest', rasterized=True)
    ax.plot(electrodes[:, 0], electrodes[:, 2], color='#161B22', lw=1.5, zorder=5)
    surface = lambda x: float(np.interp(x, electrodes[:, 0], electrodes[:, 2]))
    for x, label, ha in marks:
        if label == 'column':
            column_ring(ax, x, surface(x))
        else:
            crossing_tick(ax, x, surface(x), label, ha)
    for x0, x1 in brackets:
        top = max(surface(x0), surface(x1)) + 1.2
        ax.plot([x0, x0, x1, x1], [top - 0.8, top, top, top - 0.8], color='#111111', lw=1.2, clip_on=False)
    ax.set_xlim(0, 188); ax.set_ylim(267, 300)
    ax.set_xlabel('Distance (m)'); ax.set_ylabel('Elevation (m)')
    ax.xaxis.set_major_locator(MultipleLocator(40)); ax.yaxis.set_major_locator(MultipleLocator(10))
    bold_axes(ax)
    vertical_colorbar(fig, cax, mesh, r'Resistivity ($\Omega$ m)', RHO_TICKS)
    return save_png(fig, filename)

## a | ERT2 and SRT2 sections

In [ ]:
draw_ert_section('ERT2', 'Fig7a_ERT2_section.png',
                 marks=[(COLUMN_ON_ERT2, 'column', None),
                        (ERT2_MEETS_ERT1, 'ERT1', 'right'),
                        (ERT2_MEETS_SRT2, 'SRT2', 'left')])

In [ ]:
srt_centres = np.asarray(SRT2['mesh']['cell_centers_m'], float)[:, :2]
sensors = SRT2['sensors']
xg, zg, image = section_image(srt_centres, SRT2['values'], SRT2['touched'] & np.isfinite(SRT2['values']),
                              sensors[:, 0], sensors[:, 2], (0, 92), (271, 298), log=False)
fig = plt.figure(figsize=(8.0, 2.7))
ax = fig.add_axes([0.11, 0.22, 0.68, 0.56])
cax = fig.add_axes([0.82, 0.22, 0.032, 0.56])
mesh = ax.pcolormesh(xg, zg, image, cmap=VEL_CMAP, norm=VEL_NORM, shading='nearest', rasterized=True)
ax.plot(sensors[:, 0], sensors[:, 2], color='#161B22', lw=1.5, zorder=5)
srt_surface = lambda x: float(np.interp(x, sensors[:, 0], sensors[:, 2]))
column_ring(ax, COLUMN_ON_SRT2, srt_surface(COLUMN_ON_SRT2))
crossing_tick(ax, SRT2_MEETS_ERT1, srt_surface(SRT2_MEETS_ERT1), 'ERT1', 'right')
crossing_tick(ax, SRT2_MEETS_ERT2, srt_surface(SRT2_MEETS_ERT2), 'ERT2', 'left')
ax.set_xlim(0, 92); ax.set_ylim(271, 299)
ax.set_xlabel('Distance (m)'); ax.set_ylabel('Elevation (m)')
ax.xaxis.set_major_locator(MultipleLocator(20)); ax.yaxis.set_major_locator(MultipleLocator(10))
bold_axes(ax)
vertical_colorbar(fig, cax, mesh, r'$V_p$ (m/s)', [500, 1000, 1500, 2000])
save_png(fig, 'Fig7a_SRT2_section.png')

## b–c | TEM current and SRT2 picks

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
import case3_llano as case3
from ofag.formats.usf import read_usf
from ofag.formats.seisimager import read_first_arrivals

case3.TDEM = ROOT / 'data/case3_Llano/tdem'
case3.SRT = ROOT / 'data/case3_Llano/srt'
SWEEP_COLORS = ['#0B3C5D', '#1F77B4', '#2CA02C', '#E0762B', '#8C564B']

# Panel b: current normalisation.
sounding = read_usf(case3.TDEM / 'sounding.usf', location=case3.LOOP_CENTRE_M, z_direction=case3.LOOP_Z_DIRECTION)
sweeps = [s for s in sounding.sweeps if not s.is_noise]
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.3), sharey=False)
for s, colour in zip(sweeps, SWEEP_COLORS):
    t = s.times_s * 1e3
    ramp = case3.TDEM_RAMP_TIME_SCALE * s.ramp_time_s
    inside = s.times_s <= 3.0 * ramp
    for ax, v in ((axes[0], np.abs(s.voltages)), (axes[1], np.abs(s.voltages) / s.current_a)):
        ax.plot(t[~inside], v[~inside], 'o-', color=colour, ms=3.2, lw=1.0)
        ax.plot(t[inside], v[inside], 'o', color=colour, ms=3.2, mfc='white')
    axes[1].plot([], [], 'o-', color=colour, ms=3.2, lw=1.0, label=f'{s.current_a:g} A, {s.frequency_hz:g} Hz')
for ax, title in zip(axes, ('Raw data', 'Normalized by transmitter current')):
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Time (ms)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    bold_axes(ax)
axes[0].set_ylabel(r'$|dB_z/dt|$ (V m$^{-2}$)')
axes[1].set_ylabel(r'(V m$^{-2}$ A$^{-1}$)')
axes[1].legend(loc='upper right', fontsize=8.5, handlelength=1.4, prop={'weight': 'bold', 'size': 8.5})
fig.tight_layout(w_pad=1.6)
save_png(fig, 'Fig7b_TEM_current.png')
print('sweeps', [(s.current_a, s.frequency_hz, len(s.times_s)) for s in sweeps])

In [ ]:
# Panel c: SRT2 picks, the picker's default and the reciprocal pairs.
arrivals = read_first_arrivals(case3.SRT / 'srt2_first_arrivals.txt')
tt = arrivals.traveltime_s * 1e3
defaulted = np.isclose(arrivals.traveltime_s, case3.PICK_DEFAULT_S, atol=1e-9)
zero = arrivals.offset_m == 0
left, right = arrivals.reciprocal_pairs()
recip_error_ms = np.median(np.abs(tt[left] - tt[right])) / np.sqrt(2)
fig, ax = plt.subplots(figsize=(4.9, 3.5))
shots = np.unique(arrivals.shot_x_m)
for shot, colour in zip(shots, SWEEP_COLORS * 2):
    m = (arrivals.shot_x_m == shot) & ~zero
    order = np.argsort(arrivals.receiver_x_m[m])
    x, y = arrivals.receiver_x_m[m][order], tt[m][order].copy()
    y[np.isclose(y, 1e3 * case3.PICK_DEFAULT_S, atol=1e-6)] = np.nan   # break the line where a default was removed
    ax.plot(x, y, '-o', color=colour, ms=3, lw=1.0)
    ax.plot([shot], [0], marker='*', color=colour, ms=10, clip_on=False, zorder=5)
ax.plot(arrivals.receiver_x_m[defaulted], tt[defaulted], 'x', color='#C0392B', ms=7, mew=1.6,
        label='Erroneous picks')
measured = ~defaulted[left] & ~defaulted[right]
left, right = left[measured], right[measured]
recip_error_ms = np.median(np.abs(tt[left] - tt[right])) / np.sqrt(2)
for i, j in zip(left, right):
    ax.plot(arrivals.receiver_x_m[[i]], tt[[i]], 's', mfc='none', mec='#1F1F1F', ms=7, mew=1.0)
ax.plot([], [], 's', mfc='none', mec='#1F1F1F', ms=7, label='Reciprocal times')
ax.set_xlabel('Distance (m)'); ax.set_ylabel('Traveltime (ms)')
ax.set_ylim(0, None)
ax.legend(loc='upper left', fontsize=9, prop={'weight': 'bold', 'size': 9})
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
bold_axes(ax)
save_png(fig, 'Fig7c_SRT2_picks.png')
print(f'SRT2 picks {tt.size}, defaulted {defaulted.sum()}, zero offset {zero.sum()}, reciprocal pairs {left.size}, sigma {recip_error_ms:.2f} ms')

## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig7a_ERT2_section.png', 'Fig7a_SRT2_section.png', 'Fig7b_TEM_current.png', 'Fig7c_SRT2_picks.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
